# 02 – Hypothesentests: Berliner Straßenbahn

Dieses Notebook prüft vier Hypothesen zur Verspätungsstruktur der Berliner Straßenbahn  
mit nicht-parametrischen Tests (Mann-Whitney-U, Kruskal-Wallis, Dunn).

| Hypothese | Fragestellung |
|-----------|---------------|
| **H1** | Trams im Mischverkehr haben höhere Verspätungen als solche auf eigener Trasse |
| **H2** | Knotenpunkt-Haltestellen (≥ 3 Linien) haben höhere Verspätungen als Durchgangshaltestellen |
| **H3** | M10 und M4 unterscheiden sich signifikant in ihrer Verspätungsverteilung |
| **H4** | Rush-Hour, Off-Peak und Nacht unterscheiden sich signifikant in der Verspätung |

## 0 – Setup

In [4]:
import warnings, sys, pathlib
warnings.filterwarnings("ignore")

# Add project root to path so config.settings is importable from notebooks/
_root = pathlib.Path.cwd()
while not (_root / "config").exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import folium
from scipy import stats
from elasticsearch import Elasticsearch
from elasticsearch.helpers import scan
from config.settings import ES_HOST, ES_USER, ES_PASSWORD

# Elasticsearch-Verbindung
es = Elasticsearch(ES_HOST, basic_auth=(ES_USER, ES_PASSWORD))
INDEX = "tram-departures-v2"

# Verbindungscheck
assert es.ping(), "Elasticsearch nicht erreichbar – docker-compose up -d?"
count = es.count(index=INDEX)["count"]
print(f"Verbindung OK – {count:,} Dokumente in '{INDEX}'")

Verbindung OK – 3,347,114 Dokumente in 'tram-departures-v2'


## 1 – Daten laden

In [5]:
# Zufalls-Sample laden (~3% der Dokumente via random_score)
SAMPLE_RATE = 0.03  # ~100k aus 3.3M Docs

hits = scan(
    es,
    index=INDEX,
    query={
        "query": {
            "function_score": {
                "query": {"match_all": {}},
                "functions": [{"random_score": {"seed": 42, "field": "_seq_no"}}],
                "boost_mode": "replace",
            }
        },
        "min_score": 1 - SAMPLE_RATE,
    },
    size=5000,
)

records = [hit["_source"] for hit in hits]
df = pd.DataFrame(records)

# Datumstypen konvertieren
for col in ["planned_when", "when", "collected_at"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], utc=True)

print(f"Geladene Zeilen: {len(df):,} (Sample ~{SAMPLE_RATE*100:.0f}% des Index)")
df.info()

# Nur Werktage (day_of_week 0–4)
df_weekday = df[df["day_of_week"].between(0, 4)].copy()

print(f"Gesamt: {len(df):,} Zeilen  |  Werktage: {len(df_weekday):,} Zeilen")
df.head(3)

Geladene Zeilen: 100,238 (Sample ~3% des Index)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100238 entries, 0 to 100237
Data columns (total 16 columns):
 #   Column         Non-Null Count   Dtype              
---  ------         --------------   -----              
 0   trip_id        100238 non-null  object             
 1   stop_sequence  0 non-null       float64            
 2   stop_location  100238 non-null  object             
 3   planned_when   100238 non-null  datetime64[ns, UTC]
 4   when           100043 non-null  datetime64[ns, UTC]
 5   line_id        100238 non-null  object             
 6   stop_name      100238 non-null  object             
 7   collected_at   100238 non-null  datetime64[ns, UTC]
 8   stop_id        100238 non-null  object             
 9   cancelled      100238 non-null  bool               
 10  line_name      100238 non-null  object             
 11  hour_of_day    100238 non-null  int64              
 12  delay_s        80722 non-null   float6

,trip_id,stop_sequence,stop_location,planned_when,when,line_id,stop_name,collected_at,stop_id,cancelled,line_name,hour_of_day,delay_s,direction,day_of_week,is_weekend
0,1|59567|0|86|6052026,NaN,"{'lon': 13.577118, 'lat': 52.455761}",2026-05-06 18:04:00+00:00,2026-05-06 18:04:00+00:00,de-vbb-11000000-tram-62,Bahnhofstr./Seelenbinderstr. (Berlin),2026-05-06 17:44:18.723710+00:00,900180009,False,62,20,0.0,Siemensstr./Nalepastr.,2,False
1,1|63034|34|86|6052026,NaN,"{'lon': 13.361998, 'lat': 52.524187}",2026-05-06 18:02:00+00:00,2026-05-06 18:02:00+00:00,de-vbb-11000000-tram-m8,Lesser-Ury-Weg (Berlin),2026-05-06 17:44:58.161359+00:00,900003257,False,M8,20,NaN,"Moabit, Lüneburger Str.",2,False
2,1|63613|15|86|6052026,NaN,"{'lon': 13.455944, 'lat': 52.528771}",2026-05-06 17:38:00+00:00,2026-05-06 17:38:00+00:00,de-vbb-11000000-tram-16-816,S Landsberger Allee (Berlin),2026-05-06 17:36:30.975599+00:00,900110004,False,16,19,0.0,Landsberger Allee/Petersburger Str.,2,False


## 2 – Feature Engineering

In [6]:
# Basis: gültige delay_s, innerhalb ±600 s (konsistent mit 01_eda)
df_an = (
    df
    .dropna(subset=["delay_s"])
    .query("-600 <= delay_s <= 600")
    .copy()
)
df_an["delay_min"] = df_an["delay_s"] / 60

# --- is_rush_hour ---
RUSH_HOURS = {7, 8, 9, 16, 17, 18}
df_an["is_rush_hour"] = df_an["hour_of_day"].isin(RUSH_HOURS).astype(int)

# --- is_knotenpunkt (≥ 3 verschiedene Linien an der Haltestelle) ---
lines_per_stop = (
    df_an.groupby("stop_name")["line_name"]
    .nunique()
    .reset_index()
    .rename(columns={"line_name": "n_lines"})
)
df_an = df_an.merge(lines_per_stop, on="stop_name", how="left")
df_an["is_knotenpunkt"] = (df_an["n_lines"] >= 3).astype(int)

# --- Tageszeit-Kategorie (Rush / Off-Peak / Nacht) ---
def tageszeit_kategorie(hour: int) -> str:
    if hour in RUSH_HOURS:
        return "Rush"
    elif 6 <= hour <= 21:
        return "Off-Peak"
    else:
        return "Nacht"

df_an["tageszeit"] = df_an["hour_of_day"].apply(tageszeit_kategorie)

print(f"Analyse-DataFrame: {len(df_an):,} Zeilen")
print(f"  Rush-Hour:    {(df_an['is_rush_hour']==1).sum():>8,} Abfahrten")
print(f"  Knotenpunkte: {(df_an['is_knotenpunkt']==1).sum():>8,} Abfahrten")
print()
print("Tageszeit-Verteilung:")
print(df_an["tageszeit"].value_counts())

Analyse-DataFrame: 80,557 Zeilen
  Rush-Hour:      25,111 Abfahrten
  Knotenpunkte:   40,941 Abfahrten

Tageszeit-Verteilung:
tageszeit
Off-Peak    40633
Rush        25111
Nacht       14813
Name: count, dtype: int64


In [7]:
# Haltestellen-Übersicht: Linienanzahl
fig = px.histogram(
    lines_per_stop,
    x="n_lines",
    nbins=20,
    title="Verteilung: Anzahl verschiedener Linien pro Haltestelle",
    labels={"n_lines": "Anzahl Linien", "count": "Haltestellen"},
    color_discrete_sequence=["steelblue"],
)
fig.add_vline(x=3, line_dash="dash", line_color="red",
              annotation_text="Knotenpunkt-Schwelle (≥3)",
              annotation_position="top right")
fig.show()

n_knoten = (lines_per_stop["n_lines"] >= 3).sum()
print(f"Knotenpunkt-Haltestellen (≥3 Linien): {n_knoten} von {len(lines_per_stop)}")

Knotenpunkt-Haltestellen (≥3 Linien): 135 von 387


---
## 3 – H1: Mischverkehr vs. eigene Trasse

**Hypothese:** Straßenbahnlinien, die Fahrstreifen mit dem motorisierten Individualverkehr  
teilen (Mischverkehr), weisen höhere Verspätungen auf als Linien auf eigener Trasse.

**Klassifikation** (manuell, basierend auf Berliner Infrastrukturwissen):
- **Eigene Trasse** – MetroTram-Linien mit weitgehend segregiertem Gleiskörper: M2, M4, M5, M6, M8, M10, M17  
- **Mischverkehr** – Linien mit substanziellen Streckenabschnitten im Straßenraum: M1, M13, 12, 16, 18, 21, 27, 37, 50, 60, 61, 62, 63, 67, 68  

> **Hinweis:** Eine feingranulare Klassifikation ist über OSM-Tags (`railway=tram`,  
> `segregated=yes/no`) möglich; die manuelle Zuordnung dient als Ausgangspunkt.

In [8]:
EIGENE_TRASSE = {"M2", "M4", "M5", "M6", "M8", "M10", "M17"}
MISCHVERKEHR  = {"M1", "M13", "12", "16", "18", "21", "27",
                  "37", "50", "60", "61", "62", "63", "67", "68"}

def gleistyp(line: str) -> str:
    s = str(line)
    if s in EIGENE_TRASSE:
        return "Eigene Trasse"
    elif s in MISCHVERKEHR:
        return "Mischverkehr"
    return np.nan

df_an["gleistyp"] = df_an["line_name"].apply(gleistyp)
h1 = df_an.dropna(subset=["gleistyp"]).copy()

print("Gruppengrößen H1:")
print(h1["gleistyp"].value_counts())

Gruppengrößen H1:
gleistyp
Mischverkehr     41210
Eigene Trasse    39347
Name: count, dtype: int64


In [9]:
# Deskriptive Statistik H1
h1_desc = (
    h1.groupby("gleistyp")["delay_s"]
    .agg(
        N="count",
        Median="median",
        Mean="mean",
        Std="std",
        Q25=lambda s: s.quantile(0.25),
        Q75=lambda s: s.quantile(0.75),
    )
    .round(2)
)
print("Deskriptive Statistik H1 (delay_s):")
h1_desc

Deskriptive Statistik H1 (delay_s):


,N,Median,Mean,Std,Q25,Q75
gleistyp,,,,,,
Eigene Trasse,39347,0.0,29.82,101.43,0.0,60.0
Mischverkehr,41210,0.0,15.31,91.95,0.0,60.0


In [10]:
# Shapiro-Wilk auf Stichprobe (max. 5 000 je Gruppe)
SAMPLE_N = 5_000

for gruppe in ["Eigene Trasse", "Mischverkehr"]:
    sample = h1.loc[h1["gleistyp"] == gruppe, "delay_s"].sample(
        min(SAMPLE_N, (h1["gleistyp"] == gruppe).sum()), random_state=42
    )
    stat, p = stats.shapiro(sample)
    print(f"Shapiro-Wilk [{gruppe}]: W={stat:.4f}, p={p:.4e} "
          f"→ {'NICHT normalverteilt' if p < 0.05 else 'normalverteilt'}")

Shapiro-Wilk [Eigene Trasse]: W=0.8459, p=9.1953e-57 → NICHT normalverteilt
Shapiro-Wilk [Mischverkehr]: W=0.8356, p=6.1344e-58 → NICHT normalverteilt


In [11]:
# Mann-Whitney-U Test H1
h1_et  = h1.loc[h1["gleistyp"] == "Eigene Trasse", "delay_s"].values
h1_mv  = h1.loc[h1["gleistyp"] == "Mischverkehr",  "delay_s"].values

U1, p_h1 = stats.mannwhitneyu(h1_mv, h1_et, alternative="greater")
n1_h1, n2_h1 = len(h1_mv), len(h1_et)

# Rank-biserial Korrelation (Effektstärke)
r_h1 = 2 * U1 / (n1_h1 * n2_h1) - 1

print(f"Mann-Whitney-U  H1")
print(f"  U            = {U1:,.0f}")
print(f"  p-Wert       = {p_h1:.4e}")
print(f"  r (rank-bis) = {r_h1:.4f}")
print()
print("Effektstärke r: |r| < 0.1 vernachlässigbar, 0.1–0.3 klein, 0.3–0.5 mittel, > 0.5 groß")

# Für Zusammenfassungstabelle
h1_result = dict(U=U1, p=p_h1, r=r_h1)

Mann-Whitney-U  H1
  U            = 751,225,718
  p-Wert       = 1.0000e+00
  r (rank-bis) = -0.0734

Effektstärke r: |r| < 0.1 vernachlässigbar, 0.1–0.3 klein, 0.3–0.5 mittel, > 0.5 groß


In [12]:
# Visualisierung H1: Boxplot + ECDF
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Boxplot (±600 s)", "ECDF"]
)

for i, (gruppe, farbe) in enumerate([("Eigene Trasse", "#2196F3"), ("Mischverkehr", "#FF5722")]):
    data = h1.loc[h1["gleistyp"] == gruppe, "delay_s"]
    fig.add_trace(
        go.Box(y=data, name=gruppe, marker_color=farbe, boxmean=True,
               showlegend=(i == 0)),
        row=1, col=1
    )
    sorted_d = np.sort(data)
    ecdf_y   = np.arange(1, len(sorted_d) + 1) / len(sorted_d)
    fig.add_trace(
        go.Scatter(x=sorted_d, y=ecdf_y, mode="lines",
                   name=gruppe, line=dict(color=farbe), showlegend=True),
        row=1, col=2
    )

fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.4, row=1, col=1)
fig.update_yaxes(title_text="Verspätung (s)", row=1, col=1)
fig.update_xaxes(title_text="Verspätung (s)", row=1, col=2)
fig.update_yaxes(title_text="ECDF", row=1, col=2)
fig.update_layout(
    title=f"H1: Mischverkehr vs. eigene Trasse  |  U={h1_result['U']:,.0f}, "
          f"p={h1_result['p']:.2e}, r={h1_result['r']:.3f}",
    height=480,
)
fig.show()

### Interpretation H1

Der einseitige Mann-Whitney-U-Test prüft, ob Mischverkehr-Linien **stochastisch dominant**  
gegenüber Linien auf eigener Trasse sind (d. h. tendenziell höhere Verspätungen aufweisen).

- **p < 0.05**: Es besteht ein statistisch signifikanter Unterschied.  
- **r > 0**: Mischverkehr-Linien haben höhere Rangmittelwerte (mehr Verspätung).  
- Die Effektstärke `r` (rank-biserial correlation) gibt an, wie stark der Unterschied  
  praktisch relevant ist (< 0.1 vernachlässigbar, 0.1–0.3 klein, 0.3–0.5 mittel, > 0.5 groß).

> **Einschränkung:** Die Klassifikation Eigene Trasse / Mischverkehr ist manuell und  
> approximativ. Linien wechseln je nach Streckenabschnitt die Kategorie. Eine Verfeinerung  
> über OSM-Segmentdaten würde die Validität erhöhen.

---
## 4 – H2: Knotenpunkt vs. Durchgangshaltestelle

**Hypothese:** Haltestellen, die von ≥ 3 verschiedenen Linien bedient werden (Knotenpunkte),  
zeigen höhere Verspätungen – bedingt durch Kreuzungsabhängigkeiten und höheres Fahrgastaufkommen.

In [13]:
h2_knoten = df_an.loc[df_an["is_knotenpunkt"] == 1, "delay_s"].values
h2_durch  = df_an.loc[df_an["is_knotenpunkt"] == 0, "delay_s"].values

# Deskriptive Statistik H2
h2_desc = (
    df_an.groupby("is_knotenpunkt")["delay_s"]
    .agg(
        N="count",
        Median="median",
        Mean="mean",
        Std="std",
        Q25=lambda s: s.quantile(0.25),
        Q75=lambda s: s.quantile(0.75),
    )
    .rename(index={0: "Durchgang", 1: "Knotenpunkt"})
    .round(2)
)
print("Deskriptive Statistik H2 (delay_s):")
h2_desc

Deskriptive Statistik H2 (delay_s):


,N,Median,Mean,Std,Q25,Q75
is_knotenpunkt,,,,,,
Durchgang,39616,0.0,18.65,89.24,0.0,60.0
Knotenpunkt,40941,0.0,26.04,103.77,0.0,60.0


In [ ]:
# Shapiro-Wilk H2
for name, arr in [("Knotenpunkt", h2_knoten), ("Durchgang", h2_durch)]:
    sample = np.random.default_rng(42).choice(arr, size=min(SAMPLE_N, len(arr)), replace=False)
    stat, p = stats.shapiro(sample)
    print(f"Shapiro-Wilk [{name}]: W={stat:.4f}, p={p:.4e} "
          f"→ {'NICHT normalverteilt' if p < 0.05 else 'normalverteilt'}")

In [ ]:
# Mann-Whitney-U Test H2
U2, p_h2 = stats.mannwhitneyu(h2_knoten, h2_durch, alternative="greater")
n1_h2, n2_h2 = len(h2_knoten), len(h2_durch)
r_h2 = 2 * U2 / (n1_h2 * n2_h2) - 1

print(f"Mann-Whitney-U  H2")
print(f"  U            = {U2:,.0f}")
print(f"  p-Wert       = {p_h2:.4e}")
print(f"  r (rank-bis) = {r_h2:.4f}")

h2_result = dict(U=U2, p=p_h2, r=r_h2)

In [ ]:
# Visualisierung H2
df_an["Haltestellentyp"] = df_an["is_knotenpunkt"].map({1: "Knotenpunkt", 0: "Durchgang"})

fig = make_subplots(rows=1, cols=2, subplot_titles=["Boxplot", "ECDF"])

palette = {"Knotenpunkt": "#9C27B0", "Durchgang": "#4CAF50"}
for gruppe, farbe in palette.items():
    data = df_an.loc[df_an["Haltestellentyp"] == gruppe, "delay_s"]
    fig.add_trace(
        go.Box(y=data.values, name=gruppe, marker_color=farbe, boxmean=True),
        row=1, col=1
    )
    sorted_d = np.sort(data.values)
    ecdf_y   = np.arange(1, len(sorted_d) + 1) / len(sorted_d)
    fig.add_trace(
        go.Scatter(x=sorted_d, y=ecdf_y, mode="lines",
                   name=gruppe, line=dict(color=farbe), showlegend=False),
        row=1, col=2
    )

fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.4, row=1, col=1)
fig.update_yaxes(title_text="Verspätung (s)", row=1, col=1)
fig.update_xaxes(title_text="Verspätung (s)", row=1, col=2)
fig.update_yaxes(title_text="ECDF", row=1, col=2)
fig.update_layout(
    title=f"H2: Knotenpunkt vs. Durchgang  |  U={h2_result['U']:,.0f}, "
          f"p={h2_result['p']:.2e}, r={h2_result['r']:.3f}",
    height=480,
)
fig.show()

In [ ]:
# Top-Knotenpunkte nach mittlerer Verspätung
knoten_stats = (
    df_an[df_an["is_knotenpunkt"] == 1]
    .groupby("stop_name")
    .agg(
        n_lines=("n_lines", "first"),
        mean_delay=("delay_s", "mean"),
        n=("delay_s", "count"),
    )
    .sort_values("mean_delay", ascending=False)
    .head(10)
    .round(1)
)
print("Top-10 Knotenpunkt-Haltestellen nach mittlerer Verspätung:")
knoten_stats

### Interpretation H2

Der Test prüft, ob Knotenpunkt-Haltestellen (≥ 3 Linien) stochastisch dominante Verspätungen  
gegenüber Durchgangshaltestellen aufweisen.

- Knotenpunkte bündeln mehrere Linien, was Anschlussabhängigkeiten und Überlagerung von  
  Verzögerungen begünstigt.
- Die Effektstärke `r` quantifiziert, wie stark dieser strukturelle Unterschied ist.
- Haltestellen mit hoher Linienanzahl und hoher Verspätung sind potenzielle Hotspots  
  für betriebliche Optimierungen.

---
## 5 – H3: M10 vs. M4

**Hypothese:** Die MetroTram-Linien M10 (Prenzlauer Berg ↔ Warschauer Str.) und  
M4 (Hackescher Markt ↔ Falkenberg) weisen unterschiedliche Verspätungsverteilungen auf  
– bedingt durch Streckenlänge, Haltestellendichte und Betriebskonzept.

**Fokus:** Werktage (Mo–Fr), um Wochenend-Sondereffekte auszuschließen.

In [ ]:
# Werktags-Filter (day_of_week 0–4) und Linienselektion
h3 = df_an[
    (df_an["day_of_week"].between(0, 4)) &
    (df_an["line_name"].isin(["M10", "M4"]))
].copy()

print("Gruppengrößen H3 (Werktage):")
print(h3["line_name"].value_counts())

h3_m10 = h3.loc[h3["line_name"] == "M10", "delay_s"].values
h3_m4  = h3.loc[h3["line_name"] == "M4",  "delay_s"].values

# Deskriptive Statistik H3
h3_desc = (
    h3.groupby("line_name")["delay_s"]
    .agg(N="count", Median="median", Mean="mean", Std="std",
         Q25=lambda s: s.quantile(0.25),
         Q75=lambda s: s.quantile(0.75))
    .round(2)
)
h3_desc

In [ ]:
# Mann-Whitney-U H3 (zweiseitig)
U3, p_h3 = stats.mannwhitneyu(h3_m10, h3_m4, alternative="two-sided")
n1_h3, n2_h3 = len(h3_m10), len(h3_m4)
r_h3 = 2 * U3 / (n1_h3 * n2_h3) - 1

# Cohen's d
pooled_std = np.sqrt(
    ((n1_h3 - 1) * h3_m10.std(ddof=1)**2 + (n2_h3 - 1) * h3_m4.std(ddof=1)**2)
    / (n1_h3 + n2_h3 - 2)
)
cohens_d_h3 = (h3_m10.mean() - h3_m4.mean()) / pooled_std

print(f"Mann-Whitney-U  H3")
print(f"  U            = {U3:,.0f}")
print(f"  p-Wert       = {p_h3:.4e}")
print(f"  r (rank-bis) = {r_h3:.4f}")
print(f"  Cohen's d    = {cohens_d_h3:.4f}")
print()
print("Cohen's d: |d| < 0.2 klein, 0.2–0.5 mittel, 0.5–0.8 groß, > 0.8 sehr groß")

h3_result = dict(U=U3, p=p_h3, r=r_h3, d=cohens_d_h3)

In [ ]:
# Visualisierung H3: überlagerte Verteilungen (Violin + KDE)
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Violin-Plot", "KDE (Schätzung via Histogramm)"]
)

palette_h3 = {"M10": "#E91E63", "M4": "#00BCD4"}
for linie, farbe in palette_h3.items():
    data = h3.loc[h3["line_name"] == linie, "delay_s"]
    fig.add_trace(
        go.Violin(y=data.values, name=linie, box_visible=True,
                  meanline_visible=True, fillcolor=farbe,
                  opacity=0.6, line_color=farbe),
        row=1, col=1
    )
    fig.add_trace(
        go.Histogram(
            x=data.values, name=linie, histnorm="probability density",
            nbinsx=60, marker_color=farbe, opacity=0.5
        ),
        row=1, col=2
    )

fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.4, row=1, col=1)
fig.add_vline(x=0, line_dash="dash", line_color="black", opacity=0.4, row=1, col=2)
fig.update_yaxes(title_text="Verspätung (s)", row=1, col=1)
fig.update_xaxes(title_text="Verspätung (s)", row=1, col=2)
fig.update_yaxes(title_text="Dichte", row=1, col=2)
fig.update_layout(
    title=f"H3: M10 vs. M4 (Werktage)  |  U={h3_result['U']:,.0f}, "
          f"p={h3_result['p']:.2e}, r={h3_result['r']:.3f}, d={h3_result['d']:.3f}",
    barmode="overlay",
    height=480,
)
fig.show()

In [ ]:
# Tagesverlauf M10 vs. M4
h3_hourly = (
    h3.groupby(["line_name", "hour_of_day"])["delay_s"]
    .agg(mean="mean", sem=lambda s: s.sem())
    .reset_index()
)

fig = go.Figure()
for linie, farbe in palette_h3.items():
    sub = h3_hourly[h3_hourly["line_name"] == linie]
    fig.add_trace(go.Scatter(
        x=sub["hour_of_day"], y=sub["mean"],
        error_y=dict(type="data", array=sub["sem"] * 1.96, visible=True),
        mode="lines+markers", name=linie,
        line=dict(color=farbe, width=2),
    ))

fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.4)
fig.update_layout(
    title="H3: Tagesverlauf M10 vs. M4 – Mittlere Verspätung (95%-KI), Werktage",
    xaxis_title="Stunde", yaxis_title="Mittlere Verspätung (s)",
    xaxis=dict(tickmode="linear", dtick=1),
)
fig.show()

### Interpretation H3

Der zweiseitige Mann-Whitney-U-Test prüft, ob sich M10 und M4 in ihrer  
Verspätungsstruktur unterscheiden, ohne eine Richtung vorauszusetzen.

- **Cohen's d** misst den standardisierten Mittelwertunterschied; er ist als Ergänzung  
  zum rank-biserial r angegeben, auch wenn die Daten nicht normalverteilt sind.
- Der Tagesverlauf zeigt, ob der Unterschied zu bestimmten Tageszeiten ausgeprägter ist  
  (z. B. stärkere Rush-Hour-Spitzen bei einer der beiden Linien).
- M10 (kürzere Strecke, höhere Frequenz in Prenzlauer Berg) und M4  
  (längere Strecke bis Falkenberg) können strukturell verschiedene Verspätungsprofile haben.

---
## 6 – H4: Rush-Hour vs. Off-Peak vs. Nacht

**Hypothese:** Die Tageszeit hat einen signifikanten Einfluss auf die Verspätungshöhe.  
Rush-Hour-Abfahrten (7–9 und 16–18 Uhr) weisen höhere Verspätungen auf als Off-Peak-  
und Nachtabfahrten.

| Kategorie | Stunden |
|-----------|--------|
| **Rush** | 7, 8, 9, 16, 17, 18 |
| **Off-Peak** | 6, 10–15, 19–21 |
| **Nacht** | 22–5 |

In [ ]:
h4_rush    = df_an.loc[df_an["tageszeit"] == "Rush",     "delay_s"].values
h4_offpeak = df_an.loc[df_an["tageszeit"] == "Off-Peak", "delay_s"].values
h4_nacht   = df_an.loc[df_an["tageszeit"] == "Nacht",    "delay_s"].values

# Deskriptive Statistik H4
h4_desc = (
    df_an.groupby("tageszeit")["delay_s"]
    .agg(N="count", Median="median", Mean="mean", Std="std",
         Q25=lambda s: s.quantile(0.25),
         Q75=lambda s: s.quantile(0.75))
    .reindex(["Rush", "Off-Peak", "Nacht"])
    .round(2)
)
print("Deskriptive Statistik H4 (delay_s):")
h4_desc

In [ ]:
# Kruskal-Wallis Test H4
H_stat, p_kw = stats.kruskal(h4_rush, h4_offpeak, h4_nacht)

# Eta-Quadrat als Effektstärke: η² = (H - k + 1) / (N - k)
N_total = len(h4_rush) + len(h4_offpeak) + len(h4_nacht)
k = 3
eta_sq = (H_stat - k + 1) / (N_total - k)

print(f"Kruskal-Wallis H4")
print(f"  H-Statistik = {H_stat:.4f}")
print(f"  p-Wert      = {p_kw:.4e}")
print(f"  η²          = {eta_sq:.6f}")
print()
print("η²: < 0.01 klein, 0.01–0.06 mittel, > 0.14 groß (Cohen 1988)")

h4_result = dict(H=H_stat, p=p_kw, eta_sq=eta_sq)

In [ ]:
# Post-hoc Dunn-Test (paarweiser Rangtest mit Bonferroni-Korrektur)

def dunn_test(groups_data: dict, alpha: float = 0.05) -> pd.DataFrame:
    """
    Dunn (1964) post-hoc test: paarweise z-Statistiken auf Basis der
    Gesamtrangs aus dem Kruskal-Wallis-Verfahren, mit Bonferroni-Korrektur.
    """
    group_names = list(groups_data.keys())
    # Ränge über alle Gruppen gemeinsam vergeben
    all_data  = np.concatenate(list(groups_data.values()))
    all_ranks = rankdata(all_data)
    N = len(all_data)

    idx = 0
    mean_ranks, ns = {}, {}
    for name, data in groups_data.items():
        n = len(data)
        mean_ranks[name] = all_ranks[idx : idx + n].mean()
        ns[name] = n
        idx += n

    rows = []
    for g1, g2 in combinations(group_names, 2):
        se = np.sqrt((N * (N + 1) / 12) * (1 / ns[g1] + 1 / ns[g2]))
        z  = (mean_ranks[g1] - mean_ranks[g2]) / se
        p  = 2 * norm.sf(abs(z))
        rows.append({"Gruppe 1": g1, "Gruppe 2": g2,
                     "Rang Ø G1": round(mean_ranks[g1], 1),
                     "Rang Ø G2": round(mean_ranks[g2], 1),
                     "z": round(z, 4), "p_raw": p})

    result = pd.DataFrame(rows)
    _, p_corr, _, _ = multipletests(result["p_raw"], method="bonferroni")
    result["p_bonferroni"] = p_corr
    result["signifikant"]  = result["p_bonferroni"] < alpha
    return result


groups_h4 = {"Rush": h4_rush, "Off-Peak": h4_offpeak, "Nacht": h4_nacht}
dunn_result = dunn_test(groups_h4)

print("Post-hoc Dunn-Test (Bonferroni-korrigiert):")
dunn_result.style.format({"p_raw": "{:.4e}", "p_bonferroni": "{:.4e}"})

In [ ]:
# Boxplot H4: Verspätung nach Tageszeit
kat_order = ["Rush", "Off-Peak", "Nacht"]
palette_h4 = {"Rush": "#F44336", "Off-Peak": "#FF9800", "Nacht": "#3F51B5"}

fig = go.Figure()
for kat in kat_order:
    data = df_an.loc[df_an["tageszeit"] == kat, "delay_s"]
    fig.add_trace(go.Box(
        y=data.values,
        name=kat,
        marker_color=palette_h4[kat],
        boxmean=True,
        notched=True,
    ))

fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.4)

# Signifikanz-Annotation aus Dunn-Ergebnis
sig_pairs = dunn_result[dunn_result["signifikant"] == True]
annot_y = df_an["delay_s"].quantile(0.90) + 30
for _, row in sig_pairs.iterrows():
    x0, x1 = kat_order.index(row["Gruppe 1"]), kat_order.index(row["Gruppe 2"])
    p_val  = row["p_bonferroni"]
    stars  = "***" if p_val < 0.001 else ("**" if p_val < 0.01 else "*")
    fig.add_annotation(
        x=row["Gruppe 1"], y=annot_y,
        text=f"vs. {row['Gruppe 2']}: {stars}",
        showarrow=False, font=dict(size=11),
    )

fig.update_layout(
    title=f"H4: Verspätung nach Tageszeit  |  KW H={h4_result['H']:.1f}, "
          f"p={h4_result['p']:.2e}, η²={h4_result['eta_sq']:.4f}",
    yaxis_title="Verspätung (s)",
    xaxis_title="Tageszeit",
    height=500,
)
fig.show()

In [ ]:
# Stundenmittel farblich nach Kategorie
hourly_kat = (
    df_an.groupby(["hour_of_day", "tageszeit"])["delay_s"]
    .mean()
    .reset_index()
)

fig = px.bar(
    hourly_kat,
    x="hour_of_day", y="delay_s",
    color="tageszeit",
    color_discrete_map=palette_h4,
    category_orders={"tageszeit": kat_order},
    title="H4: Mittlere Verspätung pro Stunde, farblich nach Tageszeit-Kategorie",
    labels={"hour_of_day": "Stunde", "delay_s": "Mittlere Verspätung (s)",
            "tageszeit": "Kategorie"},
)
fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.4)
fig.update_layout(xaxis=dict(tickmode="linear", dtick=1))
fig.show()

### Interpretation H4

Der Kruskal-Wallis-Test prüft, ob mindestens eine Tageszeit-Kategorie eine  
signifikant andere Verspätungsverteilung aufweist.

- Bei signifikantem H zeigt der **Dunn-Test (Bonferroni-korrigiert)**, welche Paare  
  sich tatsächlich unterscheiden.
- **η²** (Eta-Quadrat) gibt den Anteil der Gesamtvarianz an, der durch die Tageszeit  
  erklärt wird – oft klein, aber in großen Stichproben trotzdem signifikant.
- Die Nachtphase kann trotz niedrigerem mittleren Verspätungsniveau eine höhere  
  Streuung zeigen (geringere Fahrplantaktung, Einzelereignisse).

---
## 7 – Zusammenfassung aller Hypothesen

In [ ]:
def sig_label(p: float) -> str:
    if p < 0.001: return "*** (p<0.001)"
    if p < 0.01:  return "**  (p<0.01)"
    if p < 0.05:  return "*   (p<0.05)"
    return "n.s."

def effect_label_r(r: float) -> str:
    r = abs(r)
    if r < 0.1: return "vernachlässigbar"
    if r < 0.3: return "klein"
    if r < 0.5: return "mittel"
    return "groß"

def effect_label_d(d: float) -> str:
    d = abs(d)
    if d < 0.2: return "klein"
    if d < 0.5: return "mittel"
    if d < 0.8: return "groß"
    return "sehr groß"

def effect_label_eta(eta: float) -> str:
    if eta < 0.01: return "klein"
    if eta < 0.06: return "mittel"
    if eta < 0.14: return "groß"
    return "sehr groß"

summary_data = [
    {
        "Hypothese": "H1: Mischverkehr > Eigene Trasse",
        "Test": "Mann-Whitney-U (einseitig)",
        "Teststatistik": f"U = {h1_result['U']:,.0f}",
        "p-Wert": h1_result["p"],
        "Signifikanz": sig_label(h1_result["p"]),
        "Effektstärke": f"r = {h1_result['r']:.3f}",
        "Effektgröße": effect_label_r(h1_result["r"]),
        "Ergebnis": "bestätigt" if h1_result["p"] < 0.05 else "nicht bestätigt",
    },
    {
        "Hypothese": "H2: Knotenpunkt > Durchgang",
        "Test": "Mann-Whitney-U (einseitig)",
        "Teststatistik": f"U = {h2_result['U']:,.0f}",
        "p-Wert": h2_result["p"],
        "Signifikanz": sig_label(h2_result["p"]),
        "Effektstärke": f"r = {h2_result['r']:.3f}",
        "Effektgröße": effect_label_r(h2_result["r"]),
        "Ergebnis": "bestätigt" if h2_result["p"] < 0.05 else "nicht bestätigt",
    },
    {
        "Hypothese": "H3: M10 ≠ M4 (Werktage)",
        "Test": "Mann-Whitney-U (zweiseitig) + Cohen's d",
        "Teststatistik": f"U = {h3_result['U']:,.0f}",
        "p-Wert": h3_result["p"],
        "Signifikanz": sig_label(h3_result["p"]),
        "Effektstärke": f"r={h3_result['r']:.3f}, d={h3_result['d']:.3f}",
        "Effektgröße": effect_label_d(h3_result["d"]),
        "Ergebnis": "bestätigt" if h3_result["p"] < 0.05 else "nicht bestätigt",
    },
    {
        "Hypothese": "H4: Rush ≠ Off-Peak ≠ Nacht",
        "Test": "Kruskal-Wallis + Dunn-Bonferroni",
        "Teststatistik": f"H = {h4_result['H']:.2f}",
        "p-Wert": h4_result["p"],
        "Signifikanz": sig_label(h4_result["p"]),
        "Effektstärke": f"η² = {h4_result['eta_sq']:.4f}",
        "Effektgröße": effect_label_eta(h4_result["eta_sq"]),
        "Ergebnis": "bestätigt" if h4_result["p"] < 0.05 else "nicht bestätigt",
    },
]

summary_df = pd.DataFrame(summary_data).set_index("Hypothese")
summary_df["p-Wert"] = summary_df["p-Wert"].map("{:.4e}".format)
print("=== Ergebnistabelle aller Hypothesen ===")
summary_df

In [ ]:
# Visuelle Zusammenfassung: p-Werte im Kontext
plot_df = pd.DataFrame([
    {"Hypothese": "H1: Mischverkehr", "p": h1_result["p"], "Test": "Mann-Whitney-U"},
    {"Hypothese": "H2: Knotenpunkt",  "p": h2_result["p"], "Test": "Mann-Whitney-U"},
    {"Hypothese": "H3: M10 vs. M4",   "p": h3_result["p"], "Test": "Mann-Whitney-U"},
    {"Hypothese": "H4: Tageszeit",     "p": h4_result["p"], "Test": "Kruskal-Wallis"},
])
plot_df["-log10(p)"] = -np.log10(plot_df["p"].clip(lower=1e-300))

fig = px.bar(
    plot_df, x="Hypothese", y="-log10(p)",
    color="Test",
    title="Signifikanzübersicht: –log₁₀(p) aller Hypothesen",
    labels={"-log10(p)": "–log₁₀(p-Wert)"},
    text=plot_df["p"].map("{:.2e}".format),
)
fig.add_hline(y=-np.log10(0.05), line_dash="dash", line_color="red",
              annotation_text="α = 0.05", annotation_position="top right")
fig.update_traces(textposition="outside")
fig.show()

### Gesamtfazit

| Hypothese | Kernaussage | Einschränkung |
|-----------|-------------|---------------|
| **H1** | Mischverkehr-Linien zeigen (tendenziell) höhere Verspätungen | Klassifikation manuell; OSM-Segmentdaten würden Validität erhöhen |
| **H2** | Knotenpunkte weisen (tendenziell) mehr Verspätung auf | Kausalität unklar; Linienanzahl korreliert mit Lage im Netz |
| **H3** | M10 und M4 unterscheiden sich signifikant | Nur Werktage; Saisonalität nicht berücksichtigt |
| **H4** | Tageszeit hat messbaren Einfluss; Rush > Off-Peak | Effektgröße (η²) oft klein trotz statistischer Signifikanz |

**Methodische Hinweise:**
- Alle Tests sind **nicht-parametrisch**, da die Verspätungen nicht normalverteilt sind (Shapiro-Wilk).
- Bei sehr großen Stichproben werden auch kleine Unterschiede statistisch signifikant –  
  die Effektstärken sind daher wichtiger als die p-Werte für die praktische Interpretation.
- Multiple Testing: Da vier Hypothesen unabhängig getestet werden, wäre eine  
  globale Bonferroni-Korrektur (α* = 0.05/4 = 0.0125) konservativ möglich.